# Phase 5: Real-time Regime Inference

Load the trained model and make current predictions

## Setup

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load trained model from Phase 4
with open('../data/regime_ml_model.pkl', 'rb') as f:
    model_bundle = pickle.load(f)
    model = model_bundle['model']
    le = model_bundle['label_encoder']
    feature_names = model_bundle['features']

# Connect to DuckDB
con = duckdb.connect('../../../data/investing/investing.duckdb', read_only=True)

print('✓ Model loaded')
print(f'  Features: {len(feature_names)}')
print(f'  Regimes: {list(le.classes_)}')


## Build Current Feature Vector

Construct all 26 features from latest market data

In [ ]:
# Load latest SPY data
spy_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'SPY' ORDER BY date DESC LIMIT 252
""").df()
spy_df['date'] = pd.to_datetime(spy_df['date'])
spy = spy_df.sort_values('date').set_index('date')['close']

# Initialize features for current date
current_date = spy.index[-1]
feat = pd.Series(index=feature_names, dtype=float)

# 1A. SPY Momentum & Volatility
feat['ret_spy_1m'] = (spy.iloc[-1] / spy.iloc[-21] - 1) if len(spy) >= 21 else np.nan
feat['ret_spy_3m'] = (spy.iloc[-1] / spy.iloc[-63] - 1) if len(spy) >= 63 else np.nan
feat['ret_spy_6m'] = (spy.iloc[-1] / spy.iloc[-126] - 1) if len(spy) >= 126 else np.nan
feat['ret_spy_12m'] = (spy.iloc[-1] / spy.iloc[-252] - 1) if len(spy) >= 252 else np.nan

daily_returns = spy.pct_change()
feat['rvol_spy_63d'] = daily_returns.iloc[-63:].std() if len(daily_returns) >= 63 else np.nan

vol_recent = daily_returns.iloc[-21:].std()
vol_trailing = daily_returns.iloc[-126:].std()
feat['vol_ratio_spy'] = vol_recent / vol_trailing if vol_trailing > 0 else np.nan

print('SPY features built')


In [ ]:
# 1B. Asset Class Signals
symbols_to_load = {'QQQ': 'qqq', 'XLE': 'xle', 'GLD': 'gld', 'CL=F': 'oil'}
prices = {}

for symbol, name in symbols_to_load.items():
    df = con.execute(f"""
        SELECT date, close FROM silver_macro_prices
        WHERE symbol = '{symbol}' ORDER BY date DESC LIMIT 252
    """).df()
    df['date'] = pd.to_datetime(df['date'])
    prices[name] = df.sort_values('date').set_index('date')['close']

# QQQ
feat['ret_qqq_3m'] = (prices['qqq'].iloc[-1] / prices['qqq'].iloc[-63] - 1) if len(prices['qqq']) >= 63 else np.nan
feat['ret_qqq_6m'] = (prices['qqq'].iloc[-1] / prices['qqq'].iloc[-126] - 1) if len(prices['qqq']) >= 126 else np.nan
feat['qqq_spy_spread_6m'] = feat['ret_qqq_6m'] - feat['ret_spy_6m']

# XLE
feat['ret_xle_3m'] = (prices['xle'].iloc[-1] / prices['xle'].iloc[-63] - 1) if len(prices['xle']) >= 63 else np.nan
feat['ret_xle_6m'] = (prices['xle'].iloc[-1] / prices['xle'].iloc[-126] - 1) if len(prices['xle']) >= 126 else np.nan

# GLD
feat['ret_gld_3m'] = (prices['gld'].iloc[-1] / prices['gld'].iloc[-63] - 1) if len(prices['gld']) >= 63 else np.nan
feat['ret_gld_6m'] = (prices['gld'].iloc[-1] / prices['gld'].iloc[-126] - 1) if len(prices['gld']) >= 126 else np.nan

# Oil
feat['ret_oil_1m'] = (prices['oil'].iloc[-1] / prices['oil'].iloc[-21] - 1) if len(prices['oil']) >= 21 else np.nan
feat['ret_oil_3m'] = (prices['oil'].iloc[-1] / prices['oil'].iloc[-63] - 1) if len(prices['oil']) >= 63 else np.nan
feat['ret_oil_6m'] = (prices['oil'].iloc[-1] / prices['oil'].iloc[-126] - 1) if len(prices['oil']) >= 126 else np.nan

print('Asset class features built')


In [ ]:
# 1C. Market Breadth
breadth_df = con.execute("""
    SELECT date,
           AVG(CASE WHEN ret_252d > 0        THEN 1.0 ELSE 0.0 END) AS breadth_pos_12m,
           AVG(CASE WHEN ret_21d  > 0        THEN 1.0 ELSE 0.0 END) AS breadth_pos_1m,
           AVG(CASE WHEN pct_from_ath > -0.10 THEN 1.0 ELSE 0.0 END) AS breadth_near_ath,
           AVG(CASE WHEN rvol_252d > 0.30    THEN 1.0 ELSE 0.0 END) AS breadth_rvol_elevated
    FROM gold_panel
    WHERE index_type = 'S&P 500'
    GROUP BY date
    ORDER BY date DESC
    LIMIT 1
""").df()

if len(breadth_df) > 0:
    feat['breadth_pos_12m'] = breadth_df.iloc[0]['breadth_pos_12m']
    feat['breadth_pos_1m'] = breadth_df.iloc[0]['breadth_pos_1m']
    feat['breadth_near_ath'] = breadth_df.iloc[0]['breadth_near_ath']
    feat['breadth_rvol_elevated'] = breadth_df.iloc[0]['breadth_rvol_elevated']

print('Breadth features built')


In [ ]:
# 1D. Fear Gauge
vix_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'VIX' ORDER BY date DESC LIMIT 63
""").df()
vix_df['date'] = pd.to_datetime(vix_df['date'])
vix = vix_df.sort_values('date').set_index('date')['close']

feat['vix_level'] = vix.iloc[-1]
feat['vix_chg_21d'] = (vix.iloc[-1] / vix.iloc[-21] - 1) if len(vix) >= 21 else np.nan

hy_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'HY_OAS' ORDER BY date DESC LIMIT 63
""").df()
hy_df['date'] = pd.to_datetime(hy_df['date'])
hy = hy_df.sort_values('date').set_index('date')['close']

feat['hy_spread'] = hy.iloc[-1]
feat['hy_spread_chg_21d'] = (hy.iloc[-1] / hy.iloc[-21] - 1) if len(hy) >= 21 else np.nan
feat['hy_spread_chg_63d'] = (hy.iloc[-1] / hy.iloc[-63] - 1) if len(hy) >= 63 else np.nan

y10_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'TNX_10Y' ORDER BY date DESC LIMIT 1
""").df()

y3m_df = con.execute("""
    SELECT date, close FROM silver_macro_prices
    WHERE symbol = 'IRX_3M' ORDER BY date DESC LIMIT 1
""").df()

if len(y10_df) > 0 and len(y3m_df) > 0:
    feat['ted_spread'] = y10_df.iloc[0]['close'] - y3m_df.iloc[0]['close']

print('Fear gauge features built')


## Make Prediction

In [ ]:
# Prepare for prediction
X_current = pd.DataFrame([feat], columns=feature_names)

# Handle any remaining NaN
X_current = X_current.fillna(X_current.mean())

# Get prediction and probabilities
pred_encoded = model.predict(X_current)[0]
pred_regime = le.inverse_transform([pred_encoded])[0]
probabilities = model.predict_proba(X_current)[0]

print(f'\n{"="*60}')
print(f'CURRENT REGIME PREDICTION')
print(f'{"="*60}')
print(f'\nDate: {current_date.strftime("%Y-%m-%d")}')
print(f'\nPredicted Regime: {pred_regime}')
print(f'\nProbabilities:')
for regime, prob in zip(le.classes_, probabilities):
    bar = '█' * int(prob * 50)
    print(f'  {regime:15s}: {prob:6.1%} {bar}')
print(f'\nConfidence: {probabilities.max():.1%}')

print(f'\nWhat this means:')
regime_meaning = {
    'EXPANSION': 'Strong rally ahead - overweight growth/tech',
    'CONTRACTION': 'Decline ahead - overweight defensive assets',
    'RECOVERY': 'Bouncing from weakness - balanced positioning',
    'LATE_CYCLE': 'Flat/moderate - maintain base allocation'
}
print(f'  → {regime_meaning.get(pred_regime, "Unknown")}')


## Feature Contribution Analysis

In [ ]:
# Get feature importance from model
importances = model.feature_importances_

# Create contribution ranking
contributions = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
    'current_value': X_current.values[0]
}).sort_values('importance', ascending=False)

print(f'\nTop 10 Features Influencing This Prediction:')
print(f'{"="*60}')
for idx, row in contributions.head(10).iterrows():
    print(f'{row["feature"]:25s} | importance: {row["importance"]:5.3f} | value: {row["current_value"]:8.3f}')
